# SkinXAI v6 — Kaggle Versiyonu

## Notebook'u çalıştırmadan önce:

**1. Dataset ekle (sağ panelde "Add Data"):**
- `kmader/skin-cancer-mnist-ham10000` → HAM10000
- `cdeotte/jpeg-isic2019-384x384` → ISIC 2019
- `cdeotte/jpeg-melanoma-256x256` → ISIC 2020 (melanoma diversity)

**2. GPU aç:**
- Sağ panel → Settings → Accelerator → **GPU T4 x2** → Save

**3. Hücreleri sırayla çalıştır.**

Model ve grafikler `/kaggle/working/models_v6/` klasörüne kaydedilir.

### V6 değişiklikleri (v5'e göre):
- HAM10000 `akiec→ak` mapping kaldırıldı (kirli etiket)
- ISIC 2020'den 584 melanoma eklendi (diversity)
- Melanoma target: 4500 → 8000
- AK target: 2000 → 3000, augmentation HIGH_RISK grubuna alındı
- SCC target: 2500 → 3500
- Class weights: melanoma ×2.5, ak ×2.2, scc ×2.0

In [2]:
# ── 1. Kurulum ──────────────────────────────────────────────────
!pip install timm albumentations -q

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
import cv2, os, random
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, recall_score
from sklearn.utils.class_weight import compute_class_weight
import warnings; warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Cihaz: {device}')
if torch.cuda.is_available():
    print(f'GPU  : {torch.cuda.get_device_name(0)}')
else:
    print('⚠️  GPU YOK — Settings → Accelerator → GPU T4 x2 seç!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 101.3 MB/s eta 0:00:0000:01:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which 

In [9]:
# ── 2. Sabitler & Yollar ────────────────────────────────────────
HAM_DIR = '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000'
ISIC_DIR = '/kaggle/input/datasets/cdeotte/jpeg-isic2019-384x384'
SAVE_DIR  = '/kaggle/working/models_v6'
SAVE_PATH = f'{SAVE_DIR}/skinxai_v6_best.pth'
os.makedirs(SAVE_DIR, exist_ok=True)

# ISIC 2020 yolunu bul (datasets/ prefix veya standart)
_i20_candidates = [
    '/kaggle/input/datasets/cdeotte/jpeg-melanoma-256x256',
    '/kaggle/input/jpeg-melanoma-256x256',
]
ISIC20_DIR = next((p for p in _i20_candidates if os.path.exists(p)), None)

CLASS_NAMES = ['ak', 'bcc', 'bkl', 'df', 'melanoma', 'nevus', 'scc', 'vasc']
LABEL2IDX   = {n: i for i, n in enumerate(CLASS_NAMES)}
IDX2LABEL   = {i: n for n, i in LABEL2IDX.items()}
IMG_SIZE    = 224
BATCH_SIZE  = 32

# V6 sampling hedefleri
V6_TARGETS = {
    'melanoma': 8000,   # 4500 → 8000: ISIC 2020 ile diversity artırıldı
    'scc':      3500,   # 2500 → 3500: daha fazla oversample
    'ak':       3000,   # 2000 → 3000: akiec kaldırıldı, temiz ISIC 2019 ak
    'bcc':      3000,   # aynı
    'bkl':      3000,   # 2500 → 3000
    'df':       1500,   # aynı (zaten %91.4)
    'vasc':     1500,   # aynı (zaten %100)
    'nevus':    3000,   # sert cap
}

print('Dataset kontrolü:')
print(f'  HAM10000  var mı: {os.path.exists(HAM_DIR)}')
print(f'  ISIC 2019 var mı: {os.path.exists(ISIC_DIR)}')
print(f'  ISIC 2020 var mı: {ISIC20_DIR is not None} ({ISIC20_DIR})')

Dataset kontrolü:
  HAM10000  var mı: True
  ISIC 2019 var mı: True
  ISIC 2020 var mı: True (/kaggle/input/datasets/cdeotte/jpeg-melanoma-256x256)


In [10]:
# ── 3. HAM10000 Yükle ───────────────────────────────────────────
def load_ham10000(ham_dir):
    # V6 DEĞİŞİKLİK: akiec→ak mapping KALDIRILDI
    # akiec = AK + SCC in situ karışımı → kirli etiket, bkl/bcc ile karışıyordu
    # AK için temiz ISIC 2019 etiketleri kullanıyoruz
    HAM_LABEL_MAP = {
        'nv':  'nevus',
        'mel': 'melanoma',
        'bcc': 'bcc',
        'bkl': 'bkl',
        'df':  'df',
        'vasc':'vasc',
        # 'akiec' → kasıtlı olarak dahil edilmedi
    }
    img_index = {}
    for root, _, files in os.walk(ham_dir):
        for f in files:
            if f.lower().endswith(('.jpg','.jpeg','.png')):
                img_index[os.path.splitext(f)[0]] = os.path.join(root, f)
    print(f'  İndekslenen görüntü: {len(img_index)}')

    csv_candidates = [
        os.path.join(ham_dir, 'HAM10000_metadata.csv'),
        os.path.join(ham_dir, 'ham10000_metadata.csv'),
    ]
    meta = None
    for p in csv_candidates:
        if os.path.exists(p): meta = pd.read_csv(p); break
    if meta is None:
        raise FileNotFoundError(f'HAM metadata CSV bulunamadı: {os.listdir(ham_dir)[:10]}')

    records, missing, skipped = [], 0, 0
    for _, row in meta.iterrows():
        dx = str(row['dx']).lower().strip()
        if dx not in HAM_LABEL_MAP:
            skipped += 1; continue   # akiec atlanıyor
        path = img_index.get(row['image_id'])
        if path is None: missing += 1; continue
        records.append({'image_path': path, 'label': HAM_LABEL_MAP[dx], 'source': 'ham'})

    df = pd.DataFrame(records)
    print(f'HAM10000: {len(df)} görüntü  ({missing} eksik, {skipped} akiec atlandı)')
    print(df['label'].value_counts().to_string())
    return df

ham_df = load_ham10000(HAM_DIR)

  İndekslenen görüntü: 10015
HAM10000: 9688 görüntü  (0 eksik, 327 akiec atlandı)
label
nevus       6705
melanoma    1113
bkl         1099
bcc          514
vasc         142
df           115


In [11]:
# ── 4. ISIC 2019 Yükle ──────────────────────────────────────────
def load_isic2019(isic_dir):
    csv_path = os.path.join(isic_dir, 'train.csv')
    img_dir  = os.path.join(isic_dir, 'train')
    df_raw   = pd.read_csv(csv_path)
    print(f'  Kolonlar: {list(df_raw.columns[:6])}')

    # cdeotte ISIC 2019: diagnosis değerleri UPPERCASE kısaltma
    DIAG_MAP = {
        'NV':   'nevus',
        'MEL':  'melanoma',
        'BCC':  'bcc',
        'AK':   'ak',
        'SCC':  'scc',
        'BKL':  'bkl',
        'DF':   'df',
        'VASC': 'vasc',
    }

    records = []
    for _, row in df_raw.iterrows():
        label = DIAG_MAP.get(str(row.get('diagnosis', '')).strip().upper())
        if label is None:
            continue
        img_name = str(row['image_name'])
        for ext in ['', '.jpg', '.jpeg', '.png']:
            path = os.path.join(img_dir, img_name + ext)
            if os.path.exists(path):
                records.append({'image_path': path, 'label': label, 'source': 'isic'})
                break

    df = pd.DataFrame(records)
    print(f'ISIC 2019: {len(df)} görüntü')
    if len(df) > 0:
        print(df['label'].value_counts().to_string())
    else:
        print('⚠️  Görüntü bulunamadı — image_name kolonunu kontrol et')
        print(f'  CSV örnek image_name: {df_raw["image_name"].iloc[0]}')
        print(f'  Klasör örnek dosya  : {os.listdir(img_dir)[0]}')
    return df

isic_df = load_isic2019(ISIC_DIR)

  Kolonlar: ['image_name', 'patient_id', 'sex', 'age_approx', 'anatom_site_general_challenge', 'diagnosis']
ISIC 2019: 25331 görüntü
label
nevus       12875
melanoma     4522
bcc          3323
bkl          2624
ak            867
scc           628
vasc          253
df            239


In [13]:
# ── 4b. ISIC 2020 Yükle (melanoma diversity) ────────────────────
def load_isic2020(isic20_dir):
    """
    ISIC 2020 formatı:
      train.csv: image_name, diagnosis, target (0/1)
      diagnosis değerleri: melanoma, nevus, seborrheic keratosis,
                           lentigo NOS, lichenoid keratosis, solar lentigo, unknown
      Görüntüler: train/ klasörü
    """
    if isic20_dir is None:
        print('ISIC 2020 bulunamadı, atlanıyor.')
        return pd.DataFrame(columns=['image_path','label','source'])

    DIAG_MAP = {
        'melanoma':              'melanoma',
        'nevus':                 'nevus',
        'seborrheic keratosis':  'bkl',
        'lentigo nos':           'bkl',
        'lichenoid keratosis':   'bkl',
        'solar lentigo':         'bkl',
    }
    # 'unknown' ve 'cafe-au-lait macule' atlanıyor

    csv_path = os.path.join(isic20_dir, 'train.csv')
    df_raw   = pd.read_csv(csv_path)

    # Görüntü klasörünü bul
    img_dir_candidates = [
        os.path.join(isic20_dir, 'train', 'train'),
        os.path.join(isic20_dir, 'train'),
    ]
    img_dir = next((p for p in img_dir_candidates if os.path.isdir(p)), None)
    if img_dir is None:
        print(f'ISIC 2020 görüntü klasörü bulunamadı: {os.listdir(isic20_dir)}')
        return pd.DataFrame(columns=['image_path','label','source'])
    print(f'  ISIC 2020 görüntü klasörü: {img_dir}')

    records = []
    for _, row in df_raw.iterrows():
        diag  = str(row.get('diagnosis', '')).lower().strip()
        label = DIAG_MAP.get(diag)
        if label is None: continue   # 'unknown' atla
        img_name = str(row['image_name'])
        for ext in ['', '.jpg', '.jpeg']:
            path = os.path.join(img_dir, img_name + ext)
            if os.path.exists(path):
                records.append({'image_path': path, 'label': label, 'source': 'isic20'})
                break

    df = pd.DataFrame(records)
    print(f'ISIC 2020: {len(df)} görüntü')
    print(df['label'].value_counts().to_string())
    return df

isic20_df = load_isic2020(ISIC20_DIR)

  ISIC 2020 görüntü klasörü: /kaggle/input/datasets/cdeotte/jpeg-melanoma-256x256/train
ISIC 2020: 6000 görüntü
label
nevus       5193
melanoma     584
bkl          223


In [14]:
# ── 5. Birleştir → Train/Val/Test Split (80/10/10) ──────────────
combined_df = pd.concat([ham_df, isic_df, isic20_df], ignore_index=True)
combined_df['label_idx'] = combined_df['label'].map(LABEL2IDX)

print('Birleşik dağılım (kaynak bazlı):')
print(combined_df.groupby(['label','source']).size().unstack(fill_value=0).to_string())
print()
vc = combined_df['label'].value_counts()
for cls in CLASS_NAMES:
    n = vc.get(cls, 0)
    print(f'  {cls:12s}: {n:6d}')
print(f'  {"TOPLAM":12s}: {len(combined_df):6d}')

train_val, test_pool = train_test_split(
    combined_df, test_size=0.10, stratify=combined_df['label'], random_state=SEED)
train_pool, val_pool = train_test_split(
    train_val, test_size=0.111, stratify=train_val['label'], random_state=SEED)

print(f'\nTrain pool : {len(train_pool)}')
print(f'Val        : {len(val_pool)}')
print(f'Test       : {len(test_pool)}')

train_pool.to_csv(f'{SAVE_DIR}/train_pool.csv', index=False)
val_pool.to_csv(f'{SAVE_DIR}/val_df.csv',       index=False)
test_pool.to_csv(f'{SAVE_DIR}/test_df.csv',     index=False)

Birleşik dağılım (kaynak bazlı):
source     ham   isic  isic20
label                        
ak           0    867       0
bcc        514   3323       0
bkl       1099   2624     223
df         115    239       0
melanoma  1113   4522     584
nevus     6705  12875    5193
scc          0    628       0
vasc       142    253       0

  ak          :    867
  bcc         :   3837
  bkl         :   3946
  df          :    354
  melanoma    :   6219
  nevus       :  24773
  scc         :    628
  vasc        :    395
  TOPLAM      :  41019

Train pool : 32819
Val        : 4098
Test       : 4102


In [16]:
# ── 6. Dengeli Sampling ─────────────────────────────────────────
def build_balanced(df, targets, seed=42):
    parts = []
    print(f'  {"Sınıf":12s} {"Mevcut":>7} {"Hedef":>7} {"İşlem":>18}')
    print('  ' + '-'*50)
    for cls, target in sorted(targets.items()):
        sub = df[df['label'] == cls].sample(frac=1, random_state=seed)
        n   = len(sub)
        if n == 0: print(f'  {cls}: VERİ YOK'); continue
        if n >= target:
            sampled = sub.iloc[:target]; op = 'cap'
        else:
            reps = target // n; rem = target % n
            sampled = pd.concat([sub]*reps + [sub.iloc[:rem]], ignore_index=True)
            op = f'oversample ×{target/n:.1f}'
        parts.append(sampled)
        print(f'  {cls:12s} {n:7d} {target:7d} {op:>18}')
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)

print('v6 Train seti:')
train_df = build_balanced(train_pool, V6_TARGETS)
train_df['label_idx'] = train_df['label'].map(LABEL2IDX)
val_pool  = val_pool.copy();  val_pool['label_idx']  = val_pool['label'].map(LABEL2IDX)
test_pool = test_pool.copy(); test_pool['label_idx'] = test_pool['label'].map(LABEL2IDX)
print(f'\nToplam train: {len(train_df)}')

v6 Train seti:
  Sınıf         Mevcut   Hedef              İşlem
  --------------------------------------------------
  ak               693    3000    oversample ×4.3
  bcc             3070    3000                cap
  bkl             3157    3000                cap
  df               284    1500    oversample ×5.3
  melanoma        4976    8000    oversample ×1.6
  nevus          19821    3000                cap
  scc              502    3500    oversample ×7.0
  vasc             316    1500    oversample ×4.7

Toplam train: 26500


In [13]:
# ── 7. Augmentation ─────────────────────────────────────────────
# V6 DEĞİŞİKLİK: ak HIGH_RISK grubuna alındı
# Confusion matrix'te ak→bcc karışımı renk benzerliğinden kaynaklanıyor
NORM = A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
S    = IMG_SIZE

AUG_HIGH_RISK = A.Compose([   # Melanoma + SCC + AK: renk ağırlıklı
    A.Resize(S,S), A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
    A.Rotate(limit=90, p=0.7),
    A.HueSaturationValue(hue_shift_limit=25, sat_shift_limit=40, val_shift_limit=20, p=0.75),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.75),
    A.CLAHE(clip_limit=4.0, tile_grid_size=(8,8), p=0.5),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.3, hue=0.1, p=0.5),
    A.CoarseDropout(max_holes=6, max_height=S//8, max_width=S//8, fill_value=0, p=0.3),
    NORM, ToTensorV2(),
])

AUG_MINORITY = A.Compose([    # DF, VASC: geometrik ağırlıklı
    A.Resize(S,S), A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
    A.Rotate(limit=180, p=0.8),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.7),
    A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=35, val_shift_limit=20, p=0.6),
    A.OneOf([A.GridDistortion(p=1.0), A.ElasticTransform(alpha=120, sigma=6, p=1.0)], p=0.4),
    A.CoarseDropout(max_holes=8, max_height=S//8, max_width=S//8, fill_value=0, p=0.35),
    NORM, ToTensorV2(),
])

AUG_MEDIUM = A.Compose([      # BCC, BKL: standart
    A.Resize(S,S), A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.3),
    A.Rotate(limit=90, p=0.5), A.RandomBrightnessContrast(p=0.5),
    A.HueSaturationValue(p=0.4), NORM, ToTensorV2(),
])

AUG_NEVUS = A.Compose([       # Nevus: minimal
    A.Resize(S,S), A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.3), NORM, ToTensorV2(),
])

AUG_VAL = A.Compose([A.Resize(S,S), NORM, ToTensorV2()])

CLASS_AUG = {
    'melanoma': AUG_HIGH_RISK,
    'scc':      AUG_HIGH_RISK,
    'ak':       AUG_HIGH_RISK,   # V6: HIGH_RISK grubuna taşındı
    'df':       AUG_MINORITY,
    'vasc':     AUG_MINORITY,
    'bcc':      AUG_MEDIUM,
    'bkl':      AUG_MEDIUM,
    'nevus':    AUG_NEVUS,
}
print('Augmentation hazır.')
print('  HIGH_RISK (renk): melanoma, scc, ak')
print('  MINORITY  (geo) : df, vasc')
print('  MEDIUM          : bcc, bkl')
print('  MINIMAL         : nevus')

Augmentation hazır.
  HIGH_RISK (renk): melanoma, scc, ak
  MINORITY  (geo) : df, vasc
  MEDIUM          : bcc, bkl
  MINIMAL         : nevus


In [14]:
# ── 8. Dataset & DataLoader ─────────────────────────────────────
def mixup_data(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0)).to(x.device)
    return lam*x + (1-lam)*x[idx], y, y[idx], lam

def mixup_criterion(crit, pred, y_a, y_b, lam):
    return lam*crit(pred,y_a) + (1-lam)*crit(pred,y_b)


class SkinDataset(Dataset):
    def __init__(self, df, is_train=True):
        self.df = df.reset_index(drop=True)
        self.is_train = is_train
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(row['image_path'])
        if img is None: img = np.zeros((IMG_SIZE,IMG_SIZE,3), dtype=np.uint8)
        else:           img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        tfm = CLASS_AUG[row['label']] if self.is_train else AUG_VAL
        return tfm(image=img)['image'], int(row['label_idx'])


train_ds = SkinDataset(train_df, True)
val_ds   = SkinDataset(val_pool,  False)
test_ds  = SkinDataset(test_pool, False)

NW = 2
train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  num_workers=NW, pin_memory=True)
val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False, num_workers=NW, pin_memory=True)
test_loader  = DataLoader(test_ds,  BATCH_SIZE, shuffle=False, num_workers=NW, pin_memory=True)

print(f'Train: {len(train_ds):6d} ({len(train_loader)} batch)')
print(f'Val  : {len(val_ds):6d}')
print(f'Test : {len(test_ds):6d}')

Train:  26500 (829 batch)
Val  :   4098
Test :   4102


In [15]:
# ── 9. Model — EfficientNetV2-S, ImageNet Pretrained ────────────
class SkinModel(nn.Module):
    def __init__(self, num_classes=8, dropout=0.4):
        super().__init__()
        self.backbone = timm.create_model(
            'tf_efficientnetv2_s', pretrained=True,
            num_classes=0, global_pool='avg')
        d = self.backbone.num_features
        self.head = nn.Sequential(
            nn.BatchNorm1d(d),
            nn.Dropout(dropout),
            nn.Linear(d, 512),
            nn.SiLU(),
            nn.Dropout(dropout*0.5),
            nn.Linear(512, num_classes),
        )
    def forward(self, x):
        return self.head(self.backbone(x))

model = SkinModel(8).to(device)
print(f'Model hazır  |  {sum(p.numel() for p in model.parameters())/1e6:.1f}M parametre')

model.safetensors:   0%|          | 0.00/86.5M [00:00<?, ?B/s]

Model hazır  |  20.8M parametre


In [16]:
# ── 10. Loss & Class Weights ────────────────────────────────────
class LabelSmoothingLoss(nn.Module):
    def __init__(self, classes=8, smoothing=0.1, weight=None):
        super().__init__()
        self.smoothing = smoothing; self.cls = classes; self.weight = weight
    def forward(self, pred, target):
        c   = 1.0 - self.smoothing; s = self.smoothing / (self.cls - 1)
        oh  = torch.zeros_like(pred).scatter_(1, target.unsqueeze(1), 1)
        soh = oh * c + (1-oh) * s
        lp  = F.log_softmax(pred, dim=1)
        if self.weight is not None:
            loss = -(soh * lp * self.weight[target].unsqueeze(1)).sum(1).mean()
        else:
            loss = -(soh * lp).sum(1).mean()
        return loss

w = compute_class_weight('balanced', classes=np.arange(8), y=train_df['label_idx'].values)
# V6 DEĞİŞİKLİK: melanoma, ak, scc ağırlıkları artırıldı
w[CLASS_NAMES.index('melanoma')] *= 2.5   # 1.8 → 2.5 (en kritik sorun)
w[CLASS_NAMES.index('ak')]       *= 2.2   # 1.2 → 2.2 (akiec kaldırıldı, temiz ak)
w[CLASS_NAMES.index('scc')]      *= 2.0   # 1.5 → 2.0
w[CLASS_NAMES.index('bcc')]      *= 1.1   # aynı
w = w / w.mean()
weights_t = torch.FloatTensor(w).to(device)
criterion = LabelSmoothingLoss(8, 0.1, weights_t)

print('Sınıf ağırlıkları (v6):')
for n, wi in zip(CLASS_NAMES, w):
    print(f'  {n:12s}: {wi:.3f}  {"█"*int(wi*8)}')

Sınıf ağırlıkları (v6):
  ak          : 1.473  ███████████
  bcc         : 0.736  █████
  bkl         : 0.669  █████
  df          : 1.339  ██████████
  melanoma    : 0.628  █████
  nevus       : 0.669  █████
  scc         : 1.147  █████████
  vasc        : 1.339  ██████████


In [17]:
# ── 11. Eğitim Fonksiyonları ────────────────────────────────────
def train_epoch(model, loader, optimizer, criterion, device, mixup=True):
    model.train()
    tot_loss = cor = tot = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        if mixup and random.random() < 0.4:
            xm, ya, yb, lam = mixup_data(imgs, labels)
            out  = model(xm)
            loss = mixup_criterion(criterion, out, ya, yb, lam)
            cor += (lam*(out.argmax(1)==ya).float()+(1-lam)*(out.argmax(1)==yb).float()).sum().item()
        else:
            out  = model(imgs)
            loss = criterion(out, labels)
            cor += (out.argmax(1)==labels).sum().item()
        optimizer.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        tot_loss += loss.item()*imgs.size(0); tot += imgs.size(0)
    return tot_loss/tot, cor/tot

@torch.no_grad()
def eval_epoch(model, loader, device):
    model.eval()
    preds, labels = [], []
    for imgs, labs in loader:
        preds.extend(model(imgs.to(device)).argmax(1).cpu().numpy())
        labels.extend(labs.numpy())
    preds, labels = np.array(preds), np.array(labels)
    return ((preds==labels).mean(),
            f1_score(labels, preds, average='macro', zero_division=0),
            recall_score(labels, preds, average=None, zero_division=0),
            preds, labels)

def log(tag, ep, tot, loss, tacc, vacc, vf1, pc, ni, pat, saved=''):
    mel = pc[CLASS_NAMES.index('melanoma')]; scc = pc[CLASS_NAMES.index('scc')]
    nev = pc[CLASS_NAMES.index('nevus')];    bcc = pc[CLASS_NAMES.index('bcc')]
    mf  = '🟢' if mel>=0.65 else ('🟡' if mel>=0.55 else '🔴')
    sf  = '🟢' if scc>=0.60 else ('🟡' if scc>=0.45 else '🔴')
    print(f'[{tag} {ep:2d}/{tot}] L={loss:.4f} Acc={tacc:.3f} | '
          f'Val acc={vacc:.3f} f1={vf1:.4f} '
          f'mel={mel:.2f}{mf} scc={scc:.2f}{sf} '
          f'nev={nev:.2f} bcc={bcc:.2f} (p{ni}/{pat}){saved}')
    print(f'         mel={mel:.2f}{mf} scc={scc:.2f}{sf} ak={ak:.2f}{af} '
          f'bcc={bcc:.2f} bkl={bkl:.2f} df={df:.2f} vasc={vasc:.2f} nev={nev:.2f}')

print('Fonksiyonlar hazır.')

Fonksiyonlar hazır.


In [18]:
# ── 12. Phase 1 — Backbone Dondurulmuş (5 Epoch) ───────────────
for p in model.backbone.parameters(): p.requires_grad = False
for p in model.head.parameters():     p.requires_grad = True

opt1   = torch.optim.AdamW(model.head.parameters(), lr=1e-3, weight_decay=1e-3)
sched1 = torch.optim.lr_scheduler.OneCycleLR(
    opt1, max_lr=1e-3, steps_per_epoch=len(train_loader), epochs=5, pct_start=0.3)

best_f1 = 0.0; best_ep = 0; ni = 0; history = []
print('='*70)
print('Phase 1 — Backbone frozen | Sadece head eğitimi (5 epoch)')
print('='*70)

for ep in range(1, 6):
    tl, ta = train_epoch(model, train_loader, opt1, criterion, device, mixup=False)
    va, vf, pc, *_ = eval_epoch(model, val_loader, device)
    sched1.step()
    history.append(dict(phase=1, ep=ep, tl=tl, vf=vf, pc=pc.tolist()))
    sv = ''
    if vf > best_f1:
        best_f1=vf; best_ep=ep; ni=0
        torch.save({'epoch':ep,'model_state_dict':model.state_dict(),
                    'val_f1':vf,'per_class_recall':pc.tolist(),
                    'best_f1':best_f1,'class_names':CLASS_NAMES}, SAVE_PATH)
        sv=' ✅'
    else: ni+=1
    log('P1', ep, 5, tl, ta, va, vf, pc, ni, 3, sv)

print(f'\nPhase 1 bitti. Best Val F1: {best_f1:.4f}')

Phase 1 — Backbone frozen | Sadece head eğitimi (5 epoch)
[P1  1/5] L=1.4311 Acc=0.487 | Val acc=0.663 f1=0.4094 mel=0.23🔴 scc=0.10🔴 nev=0.82 bcc=0.79 (p0/3) ✅
[P1  2/5] L=1.2582 Acc=0.572 | Val acc=0.680 f1=0.4105 mel=0.24🔴 scc=0.11🔴 nev=0.84 bcc=0.75 (p0/3) ✅
[P1  3/5] L=1.2311 Acc=0.587 | Val acc=0.697 f1=0.4336 mel=0.21🔴 scc=0.10🔴 nev=0.86 bcc=0.74 (p0/3) ✅
[P1  4/5] L=1.2123 Acc=0.598 | Val acc=0.696 f1=0.4440 mel=0.21🔴 scc=0.11🔴 nev=0.87 bcc=0.71 (p0/3) ✅
[P1  5/5] L=1.1959 Acc=0.607 | Val acc=0.703 f1=0.4512 mel=0.22🔴 scc=0.13🔴 nev=0.87 bcc=0.75 (p0/3) ✅

Phase 1 bitti. Best Val F1: 0.4512


In [ ]:
# ── 13. Phase 2 — Tam Eğitim (30 Epoch) ────────────────────────
for p in model.parameters(): p.requires_grad = True

opt2 = torch.optim.AdamW([
    {'params': model.backbone.parameters(), 'lr': 1e-5},
    {'params': model.head.parameters(),     'lr': 1e-4},
], weight_decay=1e-4)

sched2  = torch.optim.lr_scheduler.ReduceLROnPlateau(
    opt2, mode='max', factor=0.5, patience=4, min_lr=1e-8)

P2 = 30; PAT = 8; ni = 0
print('='*70)
print(f'Phase 2 — Full fine-tune | Başlangıç F1: {best_f1:.4f}')
print('='*70)

for ep in range(1, P2+1):
    tl, ta   = train_epoch(model, train_loader, opt2, criterion, device, mixup=True)
    va, vf, pc, *_ = eval_epoch(model, val_loader, device)
    sched2.step(vf)
    history.append(dict(phase=2, ep=ep, tl=tl, vf=vf, pc=pc.tolist()))
    sv = ''
    if vf > best_f1:
        best_f1=vf; best_ep=ep; ni=0
        torch.save({'epoch':ep,'model_state_dict':model.state_dict(),
                    'optimizer_state_dict':opt2.state_dict(),
                    'val_acc':va,'val_f1':vf,'per_class_recall':pc.tolist(),
                    'best_f1':best_f1,'history':history,'class_names':CLASS_NAMES}, SAVE_PATH)
        sv=' ✅'
    else: ni+=1
    log('P2', ep, P2, tl, ta, va, vf, pc, ni, PAT, sv)
    if ni >= PAT:
        print(f'\nEarly stopping ({PAT} epoch iyileşme yok).')
        break

print(f'\nPhase 2 bitti. Best F1: {best_f1:.4f} (epoch {best_ep})')

Phase 2 — Full fine-tune | Başlangıç F1: 0.4512
[P2  1/30] L=1.1952 Acc=0.611 | Val acc=0.732 f1=0.4892 mel=0.23🔴 scc=0.16🔴 nev=0.90 bcc=0.82 (p0/8) ✅
[P2  2/30] L=1.1154 Acc=0.663 | Val acc=0.744 f1=0.5266 mel=0.24🔴 scc=0.19🔴 nev=0.90 bcc=0.88 (p0/8) ✅
[P2  3/30] L=1.0658 Acc=0.689 | Val acc=0.752 f1=0.5669 mel=0.24🔴 scc=0.33🔴 nev=0.90 bcc=0.89 (p0/8) ✅
[P2  4/30] L=1.0282 Acc=0.712 | Val acc=0.759 f1=0.5621 mel=0.21🔴 scc=0.24🔴 nev=0.93 bcc=0.85 (p1/8)
[P2  5/30] L=0.9917 Acc=0.735 | Val acc=0.770 f1=0.5944 mel=0.23🔴 scc=0.30🔴 nev=0.92 bcc=0.87 (p0/8) ✅
[P2  6/30] L=0.9564 Acc=0.752 | Val acc=0.774 f1=0.6060 mel=0.23🔴 scc=0.33🔴 nev=0.93 bcc=0.90 (p0/8) ✅
[P2  7/30] L=0.9347 Acc=0.767 | Val acc=0.771 f1=0.6254 mel=0.15🔴 scc=0.37🔴 nev=0.93 bcc=0.92 (p0/8) ✅
[P2  8/30] L=0.9010 Acc=0.784 | Val acc=0.781 f1=0.6480 mel=0.24🔴 scc=0.43🔴 nev=0.93 bcc=0.92 (p0/8) ✅
[P2  9/30] L=0.8871 Acc=0.792 | Val acc=0.787 f1=0.6664 mel=0.24🔴 scc=0.46🟡 nev=0.93 bcc=0.91 (p0/8) ✅
[P2 10/30] L=0.8575 Acc=0.8

In [2]:
# ── 14. Learning Curve ──────────────────────────────────────────
p2h = [h for h in history if h['phase']==2]
if p2h:
    ep   = [h['ep']  for h in p2h]
    f1s  = [h['vf']  for h in p2h]
    mels = [h['pc'][CLASS_NAMES.index('melanoma')] for h in p2h]
    sccs = [h['pc'][CLASS_NAMES.index('scc')]      for h in p2h]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13,4))
    ax1.plot(ep, f1s,  'b-o', ms=4, label='Val Macro-F1')
    ax1.axhline(0.725, c='gray', ls='--', label='v3 baseline')
    ax1.axhline(0.80,  c='green', ls=':', label='hedef 0.80')
    ax1.set_title('Val Macro-F1'); ax1.legend(); ax1.grid(alpha=.3)

    ax2.plot(ep, mels, 'r-o', ms=4, label='melanoma')
    ax2.plot(ep, sccs, 'g-s', ms=4, label='scc')
    ax2.axhline(0.597, c='r', ls='--', alpha=.5, label='v3 mel')
    ax2.axhline(0.540, c='g', ls='--', alpha=.5, label='v3 scc')
    ax2.set_title('Melanoma & SCC Recall'); ax2.legend(); ax2.grid(alpha=.3)

    plt.tight_layout()
    plt.savefig(f'{SAVE_DIR}/curve_v6.png', dpi=150, bbox_inches='tight')
    plt.show()

NameError: name 'history' is not defined

In [ ]:
# ── 15. Test & TTA Değerlendirmesi ──────────────────────────────
ckpt = torch.load(SAVE_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
print(f'Best model: Val F1={ckpt["val_f1"]:.4f} | Epoch={ckpt["epoch"]}')

# Argmax
ta, tf, pc, tp, tl = eval_epoch(model, test_loader, device)
print(f'\nTest argmax:  Acc={ta:.4f}  F1={tf:.4f}')
print(classification_report(tl, tp, target_names=CLASS_NAMES, digits=3))

# TTA
TTA_T = [
    A.Compose([A.Resize(224,224), NORM, ToTensorV2()]),
    A.Compose([A.Resize(224,224), A.HorizontalFlip(p=1), NORM, ToTensorV2()]),
    A.Compose([A.Resize(224,224), A.VerticalFlip(p=1),   NORM, ToTensorV2()]),
    A.Compose([A.Resize(256,256), A.CenterCrop(224,224), NORM, ToTensorV2()]),
    A.Compose([A.Resize(224,224), A.Rotate(limit=15,p=1),NORM, ToTensorV2()]),
]

@torch.no_grad()
def tta_eval(model, df, transforms, device, bs=32):
    model.eval()
    all_p, all_l = [], None
    for i, tfm in enumerate(transforms):
        print(f'  TTA {i+1}/{len(transforms)}...', end='\r')
        class D(Dataset):
            def __init__(self,df,t): self.df=df.reset_index(drop=True); self.t=t
            def __len__(self): return len(self.df)
            def __getitem__(self,i):
                r=self.df.iloc[i]; img=cv2.imread(r['image_path'])
                img=np.zeros((224,224,3),dtype=np.uint8) if img is None else cv2.cvtColor(img,cv2.COLOR_BGR2RGB)
                return self.t(image=img)['image'], int(r['label_idx'])
        ld=DataLoader(D(df,tfm),bs,shuffle=False,num_workers=2,pin_memory=True)
        probs,labs=[],[]
        for imgs,l in ld:
            probs.extend(torch.softmax(model(imgs.to(device)),1).cpu().numpy())
            labs.extend(l.numpy())
        all_p.append(np.array(probs))
        if all_l is None: all_l=labs
    avg=np.mean(all_p,0); pred=np.argmax(avg,1)
    return (np.mean(pred==np.array(all_l)),
            f1_score(all_l,pred,average='macro',zero_division=0),
            pred, all_l, avg)

print('\nTTA değerlendirmesi...')
tta_acc, tta_f1, tta_p, tta_l, tta_probs = tta_eval(model, test_pool, TTA_T, device)
print(f'\nTTA:  Acc={tta_acc:.4f} ({tta_acc*100:.1f}%)  F1={tta_f1:.4f}')
print(classification_report(tta_l, tta_p, target_names=CLASS_NAMES, digits=3))

V3 = {'ak':0.877,'bcc':0.875,'bkl':0.667,'df':0.831,
      'melanoma':0.597,'nevus':0.874,'scc':0.540,'vasc':0.962}
tta_pc = recall_score(tta_l, tta_p, average=None, zero_division=0)
print('\nv3 → v6 TTA karşılaştırması:')
for i, cls in enumerate(CLASS_NAMES):
    d=(tta_pc[i]-V3[cls])*100
    print(f'  {cls:12s} {V3[cls]*100:5.1f}% → {tta_pc[i]*100:5.1f}%  ({d:+.1f}%)  {"✅" if d>2 else ("➡" if abs(d)<=2 else "❌")}')

In [ ]:
# ── 16. Threshold Optimizasyonu + Confusion Matrix ──────────────
# Test tahminlerini tazele — threshold döngüsü tp/tl değişkenlerini eziyor
# Bu satırlar hücre kaç kez çalıştırılırsa çalıştırılsın doğru sonuç verir
ckpt_loaded = torch.load(SAVE_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt_loaded['model_state_dict'])
_, _, _, test_preds_arg, test_labels_arg = eval_epoch(model, test_loader, device)
print(f'Test preds: {test_preds_arg.shape}  Labels: {test_labels_arg.shape}')

# Val üzerinde per-class threshold bul
@torch.no_grad()
def get_probs(model, df, device, bs=32):
    model.eval()
    class D(Dataset):
        def __init__(self,df): self.df=df.reset_index(drop=True)
        def __len__(self): return len(self.df)
        def __getitem__(self,i):
            r=self.df.iloc[i]; img=cv2.imread(r['image_path'])
            img=np.zeros((224,224,3),dtype=np.uint8) if img is None else cv2.cvtColor(img,cv2.COLOR_BGR2RGB)
            return AUG_VAL(image=img)['image'], int(r['label_idx'])
    ld=DataLoader(D(df),bs,shuffle=False,num_workers=2,pin_memory=True)
    probs,labs=[],[]
    for imgs,l in ld:
        probs.extend(torch.softmax(model(imgs.to(device)),1).cpu().numpy()); labs.extend(l.numpy())
    return np.array(probs), np.array(labs)

vp, vt = get_probs(model, val_pool, device)
thresholds={}
print(f'  {"Sınıf":12s} {"Threshold":>10} {"Recall":>8} {"F1":>8}')
print('  '+'-'*44)
for i, cls in enumerate(CLASS_NAMES):
    bf=0; bt=0.5
    for thr in np.arange(0.10,0.80,0.02):
        pb=(vp[:,i]>=thr).astype(int); tb=(vt==i).astype(int)
        tp=((pb==1)&(tb==1)).sum(); fp=((pb==1)&(tb==0)).sum(); fn=((pb==0)&(tb==1)).sum()
        if (tp+fp)==0 or (tp+fn)==0: continue
        pr=tp/(tp+fp); rc=tp/(tp+fn); f=2*pr*rc/(pr+rc+1e-9)
        if f>bf: bf=f; bt=thr
    thresholds[cls]=round(bt,2)
    rec=((vp[:,i]>=bt)&(vt==i)).sum()/max((vt==i).sum(),1)
    print(f'  {cls:12s} {bt:10.2f} {rec:8.3f} {bf:8.3f}')
print('\npredict.py RISK_THRESHOLDS için önerilen değerler:')
for cls in ['melanoma','bcc','scc','ak']:
    print(f'  "{cls}": {max(thresholds[cls]-0.08, 0.15):.2f},')

# Confusion matrix
fig, axes = plt.subplots(1,2,figsize=(20,8))
for ax, (preds, lbl) in zip(axes, [(test_preds_arg, test_labels_arg), (tta_p,tta_l)]):
    cm = confusion_matrix(lbl, preds)
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=CLASS_NAMES,
                yticklabels=CLASS_NAMES, cmap='Blues', linewidths=.4, ax=ax)
    ax.set_title(f'SkinXAI v6 — {"Argmax" if preds is test_preds_arg else "TTA"}',
                 fontsize=12, fontweight='bold')
    ax.set_ylabel('Gerçek'); ax.set_xlabel('Tahmin')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/cm_v6.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\n✅ Grafik kaydedildi: {SAVE_DIR}/cm_v6.png')

In [ ]:
# ── 17. Final Model — predict.py Uyumlu ────────────────────────
final_path = f'{SAVE_DIR}/skinxai_v6_final.pth'
torch.save({
    'model_state_dict':      model.state_dict(),
    'class_names':           CLASS_NAMES,
    'val_f1':                ckpt['val_f1'],
    'test_f1_tta':           float(tta_f1),
    'test_acc_tta':          float(tta_acc),
    'per_class_recall_tta':  tta_pc.tolist(),
    'thresholds':            thresholds,
}, final_path)

print(f'✅ Model kaydedildi: {final_path}')
print()
print('📥 İndirmek için sağ panelde "Output" sekmesine tıkla.')
print()
print('predict.py RISK_THRESHOLDS (bunu predict.py\'e yapıştır):')
print('RISK_THRESHOLDS = {')
for cls in ['melanoma','bcc','scc','ak']:
    thr = max(thresholds[cls]-0.08, 0.15)
    print(f'    "{cls}": {thr:.2f},')
print('}')
print()
print(f'TTA Accuracy : {tta_acc*100:.1f}%')
print(f'TTA Macro-F1 : {tta_f1:.4f}')
for i,cls in enumerate(CLASS_NAMES):
    d=(tta_pc[i]-V3[cls])*100
    print(f'  {cls:12s}: {tta_pc[i]*100:.1f}% ({d:+.1f}%)')

In [15]:
import os

v3_train_ids = set(
    pd.read_csv("/kaggle/input/datasets/beyzyilmaz/v3-train-image-ids/v3_train_image_ids.csv")["image_id"]
)
print(f"v3 train örnek sayısı: {len(v3_train_ids)}")

v6_test_ids = set(
    test_pool["image_path"].apply(lambda p: os.path.splitext(os.path.basename(p))[0])
)
v6_val_ids = set(
    val_pool["image_path"].apply(lambda p: os.path.splitext(os.path.basename(p))[0])
)

overlap_test = v3_train_ids & v6_test_ids
overlap_val  = v3_train_ids & v6_val_ids

print(f"v3 train ∩ v6 test : {len(overlap_test)} / {len(v6_test_ids)}  (%{100*len(overlap_test)/len(v6_test_ids):.1f})")
print(f"v3 train ∩ v6 val  : {len(overlap_val)} / {len(v6_val_ids)}  (%{100*len(overlap_val)/len(v6_val_ids):.1f})")

v3 train örnek sayısı: 18009
v3 train ∩ v6 test : 1793 / 4023  (%44.6)
v3 train ∩ v6 val  : 2699 / 4031  (%67.0)


In [5]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith(".csv") and "v3_train" in f:
            print(os.path.join(root, f))

/kaggle/input/datasets/beyzyilmaz/v3-train-image-ids/v3_train_image_ids.csv


In [17]:
v3_train_ids = set(
    pd.read_csv("/kaggle/input/datasets/beyzyilmaz/v3-train-image-ids/v3_train_image_ids.csv")["image_id"]
)

def clean_split(df):
    ids = df["image_path"].apply(lambda p: os.path.splitext(os.path.basename(p))[0])
    return df[~ids.isin(v3_train_ids)].reset_index(drop=True)

test_clean = clean_split(test_pool)
val_clean  = clean_split(val_pool)

print(f"test_pool: {len(test_pool)} → test_clean: {len(test_clean)}")
print(f"val_pool : {len(val_pool)} → val_clean : {len(val_clean)}")

test_pool: 4102 → test_clean: 2293
val_pool : 4098 → val_clean : 1338


In [18]:
!git clone https://github.com/beyzayilmz/SkinXAI.git
import sys
sys.path.append('/kaggle/working/SkinXAI')

Cloning into 'SkinXAI'...
remote: Enumerating objects: 110, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 110 (delta 6), reused 12 (delta 5), pack-reused 95 (from 1)
Receiving objects: 100% (110/110), 332.02 MiB | 41.97 MiB/s, done.
Resolving deltas: 100% (18/18), done.
Updating files: 100% (68/68), done.


In [19]:
import importlib
import predict
importlib.reload(predict)
from predict import predict_image, CLASS_NAMES

In [20]:
import pandas as pd
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix

V3  = "/kaggle/working/SkinXAI/skinxai_v3_best.pth"
V6  = "/kaggle/working/SkinXAI/skinxai_v6_final.pth"
EXP = "/kaggle/working/SkinXAI/skinxai_expert_final.pth"

CONFIGS = {
    "sadece_v3":        dict(model_path=V3, version="v3", ensemble=False, cascade=False),
    "sadece_v6":        dict(model_path=V6, version="v6", ensemble=False, cascade=False),
    "ensemble_v3_v6":   dict(v3_path=V3, v6_path=V6, ensemble=True, cascade=False),
    "ensemble_cascade": dict(v3_path=V3, v6_path=V6, expert_path=EXP, ensemble=True, cascade=True),
}

results = {}
final_cms = {}

for name, kwargs in CONFIGS.items():
    print(f"\n>>> {name} çalışıyor...")
    y_true, y_pred = [], []
    for i, row in test_clean.iterrows():
        out = predict_image(row["image_path"], use_tta=True, **kwargs)
        y_true.append(row["label"])
        y_pred.append(out["predicted_class"])
        if (i + 1) % 500 == 0:
            print(f"  {i+1}/{len(test_clean)}")

    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro", labels=CLASS_NAMES, zero_division=0)
    report = classification_report(y_true, y_pred, labels=CLASS_NAMES, output_dict=True, zero_division=0)

    results[name] = {
        "accuracy": acc,
        "macro_f1": macro_f1,
        "melanoma_recall": report["melanoma"]["recall"],
        "ak_recall": report["ak"]["recall"],
        "scc_recall": report["scc"]["recall"],
        "bkl_recall": report["bkl"]["recall"],
        "nevus_recall": report["nevus"]["recall"],
    }
    final_cms[name] = confusion_matrix(y_true, y_pred, labels=CLASS_NAMES)
    print(f"{name}: bitti ✅  Acc={acc:.4f}  MacroF1={macro_f1:.4f}")

comparison = pd.DataFrame(results).T
print("\n" + "="*70)
print(comparison.round(4))
comparison.to_csv("/kaggle/working/ablation_comparison_clean.csv")
print("\n✅ Kaydedildi: /kaggle/working/ablation_comparison_clean.csv")


>>> sadece_v3 çalışıyor...
[SkinXAI] v3 yüklendi ✅  |  /kaggle/working/SkinXAI/skinxai_v3_best.pth  |  Cihaz: cuda
  500/2293
  1000/2293
  1500/2293
  2000/2293
sadece_v3: bitti ✅  Acc=0.7828  MacroF1=0.4861

>>> sadece_v6 çalışıyor...
[SkinXAI] v6 yüklendi ✅  |  /kaggle/working/SkinXAI/skinxai_v6_final.pth  |  Cihaz: cuda
  500/2293
  1000/2293
  1500/2293
  2000/2293
sadece_v6: bitti ✅  Acc=0.8203  MacroF1=0.7017

>>> ensemble_v3_v6 çalışıyor...
  500/2293
  1000/2293
  1500/2293
  2000/2293
ensemble_v3_v6: bitti ✅  Acc=0.8151  MacroF1=0.6339

>>> ensemble_cascade çalışıyor...
[SkinXAI] expert yüklendi ✅  |  /kaggle/working/SkinXAI/skinxai_expert_final.pth  |  Cihaz: cuda
  500/2293
  1000/2293
  1500/2293
  2000/2293
ensemble_cascade: bitti ✅  Acc=0.8147  MacroF1=0.6279

                  accuracy  macro_f1  melanoma_recall  ak_recall  scc_recall  \
sadece_v3           0.7828    0.4861           0.5885     0.0000      0.4706   
sadece_v6           0.8203    0.7017           0.4140

In [21]:
!cd /kaggle/working/SkinXAI && git pull
import importlib
import predict
importlib.reload(predict)
from predict import predict_image, CLASS_NAMES

remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 3 (delta 2), reused 3 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 380 bytes | 380.00 KiB/s, done.
From https://github.com/beyzayilmz/SkinXAI
   44e29cd..f34554d  main       -> origin/main
Updating 44e29cd..f34554d
Fast-forward
 predict.py | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)


In [22]:
import pandas as pd
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix

V3  = "/kaggle/working/SkinXAI/skinxai_v3_best.pth"
V6  = "/kaggle/working/SkinXAI/skinxai_v6_final.pth"
EXP = "/kaggle/working/SkinXAI/skinxai_expert_final.pth"

CONFIGS_V2 = {
    "ensemble_v3_v6_fixed":   dict(v3_path=V3, v6_path=V6, ensemble=True, cascade=False),
    "ensemble_cascade_fixed": dict(v3_path=V3, v6_path=V6, expert_path=EXP, ensemble=True, cascade=True),
}

results_v2 = {}

for name, kwargs in CONFIGS_V2.items():
    print(f"\n>>> {name} çalışıyor...")
    y_true, y_pred = [], []
    for i, row in test_clean.iterrows():
        out = predict_image(row["image_path"], use_tta=True, **kwargs)
        y_true.append(row["label"])
        y_pred.append(out["predicted_class"])
        if (i + 1) % 500 == 0:
            print(f"  {i+1}/{len(test_clean)}")

    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro", labels=CLASS_NAMES, zero_division=0)
    report = classification_report(y_true, y_pred, labels=CLASS_NAMES, output_dict=True, zero_division=0)

    results_v2[name] = {
        "accuracy": acc,
        "macro_f1": macro_f1,
        "melanoma_recall": report["melanoma"]["recall"],
        "ak_recall": report["ak"]["recall"],
        "scc_recall": report["scc"]["recall"],
        "bkl_recall": report["bkl"]["recall"],
        "nevus_recall": report["nevus"]["recall"],
    }
    print(f"{name}: bitti ✅  Acc={acc:.4f}  MacroF1={macro_f1:.4f}")

comparison_v2 = pd.DataFrame(results_v2).T
print("\n" + "="*70)
print("Önceki sonuç (referans):")
print(comparison.round(4))
print("\nDüzeltilmiş sonuç:")
print(comparison_v2.round(4))


>>> ensemble_v3_v6_fixed çalışıyor...
[SkinXAI] v3 yüklendi ✅  |  /kaggle/working/SkinXAI/skinxai_v3_best.pth  |  Cihaz: cuda
[SkinXAI] v6 yüklendi ✅  |  /kaggle/working/SkinXAI/skinxai_v6_final.pth  |  Cihaz: cuda
  500/2293
  1000/2293
  1500/2293
  2000/2293
ensemble_v3_v6_fixed: bitti ✅  Acc=0.8273  MacroF1=0.6937

>>> ensemble_cascade_fixed çalışıyor...
[SkinXAI] expert yüklendi ✅  |  /kaggle/working/SkinXAI/skinxai_expert_final.pth  |  Cihaz: cuda
  500/2293
  1000/2293
  1500/2293
  2000/2293
ensemble_cascade_fixed: bitti ✅  Acc=0.8234  MacroF1=0.6699

Önceki sonuç (referans):
                  accuracy  macro_f1  melanoma_recall  ak_recall  scc_recall  \
sadece_v3           0.7828    0.4861           0.5885     0.0000      0.4706   
sadece_v6           0.8203    0.7017           0.4140     0.7143      0.7647   
ensemble_v3_v6      0.8151    0.6339           0.4763     0.2987      0.6471   
ensemble_cascade    0.8147    0.6279           0.4788     0.3506      0.6471   

       